# Nosepoke activations during the outbound path

How often does the mouse poke *before* reaching the target (the outbound leg), i.e. outside
the inbound period where the trial cue / decision poke happens? Uses the raw HARP nosepoke
device log (`nosepoke:Activations`, an 18-port beam-break signal), independent of the
events-log outcome logic, so we can see every poke with a timestamp and check whether pokes
(and the Success/Fail/Miss decision) really are inbound-only.

A poke = a beam-state **onset** (LOW->HIGH). Each onset is assigned to its trial and to the
**outbound** leg (`start_time` -> `tz_triggered_time`) or **inbound** leg (`tz_triggered_time`
-> `end_time`). Miss trials never trigger the target zone, so they have no inbound leg - the
whole trial counts as outbound (pre-cue). Ports are counted two ways: **all 18** and the
trial's **target (CorrectPort)** only.

Figures: (1) outbound pokes/trial across first/mid/last, all + target; (2) outbound vs inbound
by outcome; (3) outbound pokes vs TTT; (4) poke locations on the arena frame; (5) a
trial-by-trial animation. 4-5 reuse the Requested_plots pose/frame machinery - see the note
at the bottom.

In [ ]:
# Setup (mirrors Requested_plots: first/mid/last spec, Training tree, pose+video for figs 4-5)
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_conduit.qc import (
    qc_datastructure, training_spec, training_session_names, filter_trials,
    TRAINING_FIRST_MID_LAST_DETAIL, TRAINING_PHASE_LABELS,
)

ROOT = pathlib.Path('/media/sepi/Elements1/PathIntegrationProtocol/BonsaiOutput/Training')

In [ ]:
# Load first/mid/last sessions with the nosepoke stream (+ pose/video for the spatial figs).
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)
spec['phase'] = spec['training_day'].map(TRAINING_PHASE_LABELS)

datastructure = qc_datastructure(
    root=ROOT, depth=2, level_names=('mouseID', 'day'),
    streams=('events', 'nosepoke', 'soundcard', 'session_settings', 'video', 'dlc'),
    include=training_session_names(spec),
)
result = datastructure.load()
trials_filtered = filter_trials(result['trials'], spec)

# Confirm the Activations shape once (dims/coords can vary): inspect before extracting.
display(result['nosepoke:Activations'])

In [ ]:
# Poke-onset extraction and per-trial outbound / inbound classification.
def activations_long(result, key='nosepoke:Activations', *, port_dim=None):
    """Long-form [session, port, Time, state] from the nosepoke Activations DataArray.
    `port_dim` is the dimension holding the NP_* labels; auto-detected if None. Confirm
    against `result['nosepoke:Activations']` (the cell above) if this mis-detects."""
    df = result[key].to_dataframe(name='state').reset_index()
    if port_dim is None:
        port_dim = next(c for c in df.columns
                        if df[c].astype(str).str.startswith('NP_').any())
    df = df.rename(columns={port_dim: 'port'})
    return df[['session', 'port', 'Time', 'state']]


def poke_onsets(activations_long_df, *, high=1, session_col='session', port_col='port',
                time_col='Time', state_col='state'):
    """One row per poke ONSET (beam-state rising to HIGH). Adds integer `port_index`."""
    a = activations_long_df.sort_values([session_col, port_col, time_col]).copy()
    a['_prev'] = a.groupby([session_col, port_col])[state_col].shift()
    out = a.loc[(a[state_col] == high) & (a['_prev'] != high),
                [session_col, port_col, time_col]].reset_index(drop=True)
    out['port_index'] = out[port_col].str.split('_').str[-1].astype(int)   # 'NP_5' -> 5
    return out


def pokes_by_trial(onsets, trials, *, session_col='session', time_col='Time'):
    """Assign each onset to its trial and tag leg (outbound/inbound) + is_target.
    Miss trials (tz_triggered_time is NaN) have no inbound leg -> all counted outbound."""
    tw = (trials[[session_col, 'trial_index', 'start_time', 'end_time',
                  'outbound_end_time', 'CorrectPort']]
          .sort_values('start_time').reset_index(drop=True))
    o = onsets.sort_values(time_col).reset_index(drop=True)
    tagged = pd.merge_asof(o, tw, left_on=time_col, right_on='start_time',
                           by=session_col, direction='backward')
    tagged = tagged[(tagged[time_col] <= tagged['end_time']).fillna(False)].copy()
    cutoff = tagged['outbound_end_time'].fillna(tagged['end_time'])
    tagged['leg'] = np.where(tagged[time_col] <= cutoff, 'outbound', 'inbound')
    tagged['is_target'] = tagged['port_index'] == tagged['CorrectPort']
    return tagged


def per_trial_poke_counts(tagged, trials, *, session_col='session'):
    """Add n_<leg>_all / n_<leg>_target columns per trial (0 where a trial had no pokes)."""
    g = (tagged.groupby([session_col, 'trial_index', 'leg'])
         .agg(all_=('port_index', 'size'), target=('is_target', 'sum')).reset_index())
    wide = g.pivot_table(index=[session_col, 'trial_index'], columns='leg',
                         values=['all_', 'target'], fill_value=0)
    wide.columns = [f'n_{leg}_{"all" if k == "all_" else "target"}' for k, leg in wide.columns]
    wide = wide.reset_index()
    out = trials.merge(wide, on=[session_col, 'trial_index'], how='left')
    for c in [c for c in wide.columns if c.startswith('n_')]:
        out[c] = out[c].fillna(0).astype(int)
    return out


onsets = poke_onsets(activations_long(result))
tagged = pokes_by_trial(onsets, trials_filtered)
poke_trials = per_trial_poke_counts(tagged, trials_filtered)
poke_trials[['mouseID', 'training_day', 'trial_index', 'outcome', 'TTT',
             'n_outbound_all', 'n_outbound_target', 'n_inbound_all']].head()

In [ ]:
# Plot helpers.
DEFAULT_MOUSE_COLOURS = {
    'FLR_M01569521': '#f994b1', 'FbR_M01569522': '#e53592', 'FL_M01569519': '#950041',
    'MR_M01569515': '#73a9cf', 'MbR_M01569518': '#2e7ebc', 'MbL_M01569517': '#004cd9',
}
PHASE_TICKS = [1, 2, 3]


def plot_pokes_per_day(poke_trials, ax, count_col, title, *,
                       mouse_colours=DEFAULT_MOUSE_COLOURS, mouse_col='mouseID',
                       day_col='training_day'):
    """Mean per-trial poke count per (mouse, phase) + black group mean."""
    s = poke_trials.groupby([mouse_col, day_col])[count_col].mean().reset_index()
    for m in [m for m in mouse_colours if m in set(s[mouse_col])]:
        d = s[s[mouse_col] == m]
        ax.plot(d[day_col], d[count_col], 'o-', color=mouse_colours[m], alpha=0.55, label=m)
    gm = s.groupby(day_col)[count_col].mean()
    ax.plot(gm.index, gm.values, 'ko-', linewidth=2.5, label='group mean')
    ax.set_title(title); ax.set_xlabel('training day')
    ax.set_xticks(PHASE_TICKS)
    ax.set_xticklabels([TRAINING_PHASE_LABELS.get(t, t) for t in PHASE_TICKS])


def plot_outbound_vs_inbound(poke_trials, ax, *, outcomes=('Success', 'Failure', 'Miss')):
    """Grouped bars: mean outbound vs inbound pokes/trial per outcome."""
    m = (poke_trials.groupby('outcome')[['n_outbound_all', 'n_inbound_all']]
         .mean().reindex(outcomes))
    x = np.arange(len(m)); w = 0.38
    ax.bar(x - w / 2, m['n_outbound_all'], w, label='outbound', color='#6a5acd')
    ax.bar(x + w / 2, m['n_inbound_all'], w, label='inbound', color='#2e8b57')
    ax.set_xticks(x); ax.set_xticklabels(m.index)
    ax.set_ylabel('mean pokes / trial'); ax.set_title('Outbound vs inbound by outcome'); ax.legend()


def plot_pokes_vs_ttt(poke_trials, ax, *, count_col='n_outbound_all'):
    """Scatter of outbound pokes vs TTT with a least-squares trend."""
    d = poke_trials.dropna(subset=['TTT', count_col])
    ax.scatter(d['TTT'], d[count_col], s=10, alpha=0.35, color='#6a5acd')
    if len(d) > 2:
        sl, ic = np.polyfit(d['TTT'], d[count_col], 1)
        xr = np.array([d['TTT'].min(), d['TTT'].max()])
        ax.plot(xr, sl * xr + ic, '--', color='crimson', label=f'slope={sl:.2f}/s')
        ax.legend()
    ax.set_xlabel('TTT (s)'); ax.set_ylabel('outbound pokes / trial')
    ax.set_title('Outbound pokes vs time-to-target')

In [ ]:
# Figure 1 | Outbound pokes/trial across first/mid/last (all ports + target port).
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
plot_pokes_per_day(poke_trials, axes[0], 'n_outbound_all', 'Outbound pokes/trial (all ports)')
plot_pokes_per_day(poke_trials, axes[1], 'n_outbound_target', 'Outbound pokes/trial (target port)')
axes[0].set_ylabel('mean pokes / trial')
axes[1].legend(fontsize=7, loc='center left', bbox_to_anchor=(1.0, 0.5))
fig.tight_layout(); plt.show()

In [ ]:
# Figure 2 | Outbound vs inbound pokes by outcome.
fig, ax = plt.subplots(figsize=(6, 4.2))
plot_outbound_vs_inbound(poke_trials, ax)
plt.show()

In [ ]:
# Figure 3 | Outbound pokes vs TTT.
fig, ax = plt.subplots(figsize=(6, 4.2))
plot_pokes_vs_ttt(poke_trials, ax)
plt.show()

## Figures 4 & 5 - spatial (to add next; render only on the data PC)

Both reuse the machinery already in `Requested_plots.ipynb`, so they're straightforward
adaptations, but they need the real pose + video, so I've left them out until the core above
is confirmed on the data PC.

- **Figure 4 - poke locations on the arena.** For each outbound poke onset, take the mouse's
  **nose** position at that time (`merge_asof` the onset times onto `pose_wide`, exactly like
  the pose-by-trial cell in Requested_plots), then scatter `(nose_x, nose_y)` over the session's
  `UndistortedVideoData` frame (`_first_frame_for_session` / `_safe_first_frame`). Colour by
  target vs non-target port. This shows *where* in the arena the animal pokes before reaching
  the goal.
- **Figure 5 - trial-by-trial animation.** Adapt `figure_overlay_animation` (Requested_plots
  Fig 1d): each step adds the next trial's outbound poke locations to the arena frame, so you
  watch the poke pattern build up across a day.

Tell me to proceed and I'll wire both in from the existing Requested_plots functions.